In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Load Data**

In [6]:
pull_data = "https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3train.csv"

data_import = pd.read_csv(pull_data)

# Drop rows with missing values
data_import = data_import.dropna()

# Take a look at data
print(data_import.shape)
data_import.head()

(306, 14)


,A,B,C,D,E,F,G,H,I,J,K,L,M,N
0,0.003555,92.159562,6.658918,0.0,1.994330,35.960534,290.003375,20.973961,7.523349,2410.680571,33.715837,1965.274105,34.370406,220.134692
1,0.015362,0.000000,20.380324,0.0,1.738552,35.118264,350.939667,25.471824,15.046698,1970.894250,39.224961,1965.274105,63.081429,198.121223
2,0.015351,0.000000,20.380324,0.0,1.738552,39.296796,271.766966,25.471824,15.046698,1970.894250,39.224961,1945.121257,27.813803,318.278075
3,0.018208,0.000000,6.284173,0.0,1.697776,38.274040,203.714027,31.087615,22.570047,1808.010428,41.208245,1954.034064,20.290963,306.354113
4,0.038841,0.000000,6.284173,0.0,1.697776,39.088964,241.076425,31.087615,22.570047,1808.010428,41.208245,1965.274105,36.785997,332.036493


**Naive grid search implementation**

In [7]:
train, test = train_test_split(
    data_import,
    test_size=0.33,
    random_state=0
)

#train on all except last column
X_train = train.iloc[:, :-1]
X_test = test.iloc[:, :-1]
y_train = train.iloc[:, -1]
y_test = test.iloc[:, -1]

# all rows / columns except target column
X = data_import.iloc[:, :-1]

# only the last column
y = data_import.iloc[:, -1]

**Add pipeline**

In [8]:
pl = Pipeline([
    ("poly", PolynomialFeatures()),
    ("norm", StandardScaler()),
    ("model", TransformedTargetRegressor(
        regressor=Lasso(alpha=0.1, max_iter=100000),
        transformer=StandardScaler()
    ))
])

# train 

pl.fit(X_train, y_train)

p_train = pl.predict(X_train)
p_test = pl.predict(X_test)

**Print training values**

In [9]:
print("Training Data Summary:")
print(pd.DataFrame({
    "Actual": y_train.values[:10],
    "Predicted": p_train[:10]
}))

Training Data Summary:
       Actual   Predicted
0  212.796869  223.186635
1  194.452311  210.008063
2  108.232890  107.756652
3  200.872906  213.817496
4  301.767973  301.254032
5  458.613941  342.031627
6  218.300236  190.325270
7  207.293501  194.456691
8  173.356070  164.160149
9  293.512922  284.080131


**Print testing values**

In [10]:
print("Testing Data Summary:")
print(pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": p_test[:10]
}))

Testing Data Summary:
       Actual   Predicted
0  297.181834  276.100365
1  177.942209  186.527755
2  330.202038  326.144041
3  199.038450  184.045626
4  213.714097  246.260367
5  139.418638  188.180886
6  410.918091  413.579944
7  210.045185  239.141558
8  189.866172  192.294596
9  188.031716  202.957952


**Print Metrics to Compare against Cross Val / GridSearch**

In [11]:
mse_train = mean_squared_error(y_train, p_train)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, p_train)

mse_test = mean_squared_error(y_test, p_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, p_test)

print("Training MSE:", mse_train)
print("Training RMSE:", rmse_train)
print("Training r^2:", r2_train)

print("Testing MSE:", mse_test)
print("Testing RMSE:", rmse_test)
print("Testing r^2:", r2_test)

Training MSE: 929.2820886851895
Training RMSE: 30.484128471799707
Training r^2: 0.8671821250463024
Testing MSE: 960.0049500232946
Testing RMSE: 30.983946650213795
Testing r^2: 0.8268179149464012


**Cross validation section**

In [12]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pl, X, y, cv=kfold,
                        scoring="neg_root_mean_squared_error")

# convert scores back to positive values
rmse_scores = -scores


**RMSE Before GridSearch is Implemented**

In [13]:
print("RMSE scores: ", rmse_scores)
print("Avg RMSE: ", rmse_scores.mean())

RMSE scores:  [27.76424263 36.68169443 28.26069359 34.40492766 31.88977558]
Avg RMSE:  31.80026677549683


**Grid Search w/ Poly_degree, Lasso, & Ridge Parameter Options**

In [14]:
# set parameters
#giving options for lasso & ridge

param_grid = [
    {
        "poly__degree": [1, 2, 3],
        "model__regressor": [Lasso(max_iter=100000)],
        "model__regressor__alpha": [0.01, 0.1, 1, 10, 100]
    },
    {
        "poly__degree": [1, 2, 3],
        "model__regressor": [Ridge()],
        "model__regressor__alpha": [0.01, 0.1, 1, 10, 100]
    }
]

# grid search for best ones
grid = GridSearchCV(
    pl,
    param_grid,
    cv=kfold,
    scoring="neg_root_mean_squared_error"
)


grid.fit(X, y)

#set model variable for grader
model = grid.best_estimator_

# pull best model type 
best_regressor = grid.best_params_["model__regressor"]


**Optimal Hyperparameters found by GridSearchCV**

In [15]:
print("Best regularization type:", type(best_regressor).__name__)
print("Best degree:", grid.best_params_["poly__degree"])
print("Best alpha:", grid.best_params_["model__regressor__alpha"])
print("Best RMSE:", -grid.best_score_)

Best regularization type: Ridge
Best degree: 2
Best alpha: 1
Best RMSE: 22.872139175044815


Based on the above, all predictions are plus or minus 23 (rounded up from 22.8 above)

**Grading Section**

In [16]:
X_grading = pd.read_csv('https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3test.csv')
predicted = model.predict(X_grading)

print(predicted)
print(len(predicted))

[  336.85188999   286.03126202   249.19178984   190.44776574
   190.34278763   242.14727687   181.91847614   208.12063573
   217.08193241   166.02970454   165.83291284   169.9394229
   217.5692571    196.43889742   239.14475736   235.37651423
   211.97631885   177.06997304   245.61511725   251.26972723
   236.84749658   209.73236353   192.96539031   231.08978553
   205.98554995   129.32562694   168.6880365    253.02832978
   251.62359297   234.32691759   224.66777527   221.94364196
   240.32291458   233.56569623   228.21274213   282.38652609
   164.49948309   233.79888173   270.26973052   179.19050642
   170.31687071   232.09103031   255.45403598   234.8522688
   198.98647771   243.75164699   195.71263455   280.68654515
   176.05876045   190.63162896   109.54402008   -23.93152021
    29.68569053    50.06743992    49.21669934    18.67681781
   -66.27953087   -32.30049981   185.00272057  -112.53329545
   -55.03006352   -37.20800081   -42.87296684   104.36378585
   206.66905287   257.7916

**Final Thoughts**

Tuning the parameters is hard when you're not that familiar with the data or have a deep understanding of how the parameters work. The last module seemed easier as the housing data was more intuitive to work with and you could evaluate it with more common sense and reasoning then a from a purely mathematical standpoint like abstract data above.

The good news is you can check the output easily with the RMSE, and it's simple enough to keep playing with the parameters to see what makes the error estimate go down. I originally started with a poly degree option of 1-6, and just Lasso. The best RMSE I achieved was through adding Ridge, which GridSearch chose along with a poly degree of 2 and alpha of 1.